# Quantum Walks for Finance
$
\renewcommand{\ket}[1]{|{#1}\rangle}
\renewcommand{\bra}[1]{\langle{#1}|}
$

In [ ]:
!pip install cudaq
%pip install ipywidgets -q

In [ ]:
import cudaq
import numpy as np
import matplotlib.pyplot as plt
import random
from scipy.optimize import minimize
import itertools, math
from cudaq import spin
from typing import List, Tuple
from itertools import combinations

## Discrete Time Quantum Walks


> **Encoding Probability Distributions Problem**:  Given a discrete probability distribution $\mathcal{P}$, generate a quantum kernel (i.e., a sequence of quantum gates) that transforms an initial state into a final state whose measurement outcomes match the target distribution $\mathcal{P}$.

One approach to tackle this problem is through random walks by using a variational algorithm to search for the random walk that will generate a targeted distribution, as depicted in the animation below.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/quantum_walk_target.gif?raw=1" width="600">


Before we tackle that problem, we need to define quantum random walks.  Let's first visualize a classical random walk along a number line generating a binomial distribution. You can imagine a walker moving left or right depending on the result of a coin flip.  

Let's see this in action with the random walk widget [here](https://nvidia.github.io/cuda-q-academic/quantum-applications-to-finance/images/ss-random-walk.html). The green dot represents the walker whose movements are dictated by flips of a coin whose fairness is controlled by the probability slider. The ending position from each run of the experiment is recorded in the histogram.  Experiment by changing the number of total steps, the probabilities, and the rules for taking a step. What patterns do you notice?

Unlike classical random walks, where probabilities dictate movement, quantum walks rely on amplitudes. Interference between paths leads to a faster spread over the position space, making quantum walks useful for generating more complex probability distributions.  

### Comparing Classical Random Walks with Discrete Time Quantum Walks

Let's consider a discrete walk taking place on a line. In this scenario, we envision a walker progressing along the
x-axis in discrete increments, with the movement governed by predefined rules based on coin flips.  A classical random walk can be viewed as a decision tree, as in the diagram below.  Here, a walker begins in the center of the line. After a coin flip, the walker moves to the left or right depending on the outcome of the coin flip (e.g., flipping heads will send the walker one step to the left, while flipping tails will send the walker one step to the right).  After $n$ coin flips, we record the position of the walker.  This experiment is repeated multiple times to generate a histogram of the walker's final position.

![classical walk](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/classical-walk.png?raw=1)

A **discrete-time quantum walk (DTQW)** is the quantum analogue of a classical random walk. In this scenario, both the position of the walk and the coin are represented as quantum states, $\ket{\psi_W}$ and $\ket{\psi_C}$, respectively. Throughout this walk, the state of the walker changes according to quantum shift operators ($S_+, S_-$) which depend on the state of the coin $\ket{\psi_C}$. In particular, the $S_{-}$ operation checks if $\ket{\psi_C}$ in the state $\ket{0}$ (i.e., heads) and if so, induces a shift left operation $DEC$ on the quantum walker. Similarly, $S_{+}$ is defined for tails and shifting right. At each time step, the state of the coin changes (i.e., is "flipped") via a quantum operation ($F$), which we'll call the coin-flip operator. Unlike the classical random walk, we will not know the position of the walker until the end of the process, when the quantum state of the walker is measured.

In the diagram below, we illustrate a 2-step quantum walk where the coin-flip operator $F$ is the Hadamard gate.  Notice how the final distribution of the walker's possible positions differs from the classical case.

![coin walk](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/dtqw-superposition-coin.png?raw=1)


#### The Line that the Walker Traverses
 For our example, we consider a line that contains $16$ positions: $0,1,\cdots,15$ encoded with the computational basis states on $4$ qubits: $\ket{\bf{0}} = \ket{0000}, \ket{\bf{1}} = \ket{0001}, \ket{\bf{2}} = \ket{0010}, \cdots, \ket{\bf{15}} = \ket{1111}$.  In other words, the computational basis state $\ket{\bf{i}}$ represents the $i^{th}$ location on the line.


#### The Walker's Position State
The walker's position after $t$ time steps is given by a linear combination of the computational basis states $$\ket{\psi_W} = \sum_{i}\alpha_i(t)\ket{\bf{i}},$$ for some $\alpha_i(t)\in\mathbb{C}$ satisfying $\sum_{i}|\alpha_i(t)|^2 = 1$.  For instance, if the walker began the experiment at position $0$, then the state of the walker at time $0$ would be $\ket{\psi_W(0)}=\ket{0000}$.  

More interestingly, the walker might begin the experiment in a superposition of two or more positions, as depicted in the figure below.  In this case, the walker's initial state is $\ket{\psi_W(0)}=\frac{1}{\sqrt{2}}\ket{0010}+\frac{1}{\sqrt{2}}\ket{0011}$. The aim of this section is to encode into CUDA-Q the first step of the discrete time quantum walk drawn below.

![quantum walk](https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/dtqw-superposition-walker.png?raw=1)



### Programming a DTQW with CUDA-Q

Let's start coding up one step of the DTQW in CUDA-Q. The circuit diagram for this is the following:

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/dtqw-one-step-diagram.png?raw=1" width="600">


### Exercise  1:
The first step is to edit the code block below to create a kernel that generates the walker's initial position state: $\ket{\psi_W(0)}$ using an $H$ ($\texttt{h}$() in CUDA-Q) gate and $CNOT$ gates $\texttt{x.ctrl(control\_qubit, target\_qubit)}$.  Define a kernel that prepares the walker qubits in an equal superposition of the states $\ket{2}$ = $\ket{0010}$ and $\ket{3}$ = $\ket{0011}$.


In [ ]:
@cudaq.kernel
def initial_position(qubits : cudaq.qvector):
    """ Apply gates to the qubits to prepare the GHZ state
    Parameters
        qubits: cudaq.qvector
        qubits for the walker
    """

    # Edit the code below this line

    # Edit the code above this line

We can verify that our solution is correct by using the `cudaq.get_state` and _`.amplitudes` commands.  Additionally, in the code block below, we will create a helper function to graph the distribution of the walker's positions, as represented by the state.

In [ ]:
# Create a quantum kernel that initializes the state of the walker

num_qubits = 4

@cudaq.kernel
def walker(num_qubits : int):
    """ Kernel to initialize the state of the walker
    Parameters
        num_qubits : int
        Number of qubits for the walker
    """
    # Initialize the qubits
    qubits = cudaq.qvector(num_qubits)

    # Apply the initial state to the qubits
    initial_position(qubits)

# Get the state of the walker after applying the quantum kernel
state = cudaq.get_state(walker, 4)

# Return the amplitudes of |0010> and |0011>
state_amplitudes = state.amplitudes([[0,0,1,0], [0,0,1,1]])

# Print
precision = 4
print('Walker state array of coefficients:', np.round(np.array(state), precision))
print('Walker statevector: {} |0010> + {} |0011>'.format(np.round(state_amplitudes[0],precision), np.round(state_amplitudes[1],precision)))

# Define a function to draw the histogram of the results

def plot_results(result, num_qubits):
    """
    Plots a histogram of quantum sampling results.

    Args:
        result (dict or cudaq.SampleResult): The counts dictionary (e.g. from get_marginal_counts).
        num_qubits (int): The number of qubits analyzed (determines x-axis states).
    """
    #  Initialize x-axis with all possible bitstrings (0 to 2^N)
    #    This ensures the graph includes states with 0 probability.
    result_dictionary = {}
    for i in range(2**num_qubits):
        bitstr = bin(i)[2:].zfill(num_qubits)
        result_dictionary[bitstr] = 0

    #  Populate with data, robustly cleaning keys
    if hasattr(result, 'items'):
        for bitstr, count in result.items():
            # Robust Fix: Filter to keep ONLY '0' and '1' characters.
            # This removes spaces and invisible null terminators (\x00).
            clean_key = "".join(c for c in str(bitstr) if c in "01")

            if clean_key in result_dictionary:
                result_dictionary[clean_key] = count
            else:
                # Useful for debugging if you passed the wrong num_qubits
                print(f"Warning: Key '{clean_key}' ignored (Length {len(clean_key)} "
                      f"!= expected {num_qubits})")

    #  Prepare Lists
    x = list(result_dictionary.keys())
    y = list(result_dictionary.values())

    #  Plot
    plt.figure(figsize=(10, 5))
    plt.bar(x, y, color='#76B900')

    plt.title(f"Sampling Results ({num_qubits} Qubits)")
    plt.xlabel("States")
    plt.ylabel("Counts")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

results = cudaq.sample(walker, 4, shots_count =1000)
plot_results(results, num_qubits)

#### Walking the Line

The next step (no pun intended) is to define incrementer (`INC`) and decrementer (`DEC`) operations, which will change the state of the walker with shifts to the left and right on the number line, respectively.  We'll use these operations to build up the shift operators $S_-$ and $S_+$ which depend on the state of the coin.  First, let's define `INC` so that when applied to a basis state $\ket{x}$, the result is $\ket{(x+1)_{\mod{16}}}$ for $x\in \{0,\cdots 15\}$.  The unitary matrix below carries out this transformation:

\[ \begin{pmatrix} 0 & 0 & 0 & 0 & \cdots  \\
                        1 & 0 & 0 & 0 & \cdots \\
                        0 & 1 & 0 & 0 & \cdots \\
                        0 & 0 & 1 & 0 & \cdots \\
                        \vdots & \vdots & \vdots & \vdots & \ddots \end{pmatrix} \]

Using the `cudaq.register_operation` we can create a custom gate for this unitary matrix. The code block defines a `cudaq.kernel` to carry out the custom INC gate, and conducts a test to verify that the operation acts as expected on the zero state.  


In [ ]:
# Define a custom operation on 4 qubits for the INC unitary matrix that
# maps |x> to |x+1> mod 16 and verify that it works as expected for |0000>

num_qubits = 4

# Define the incrementer matrix
def incrementer(num_qubits):
    size = 2**num_qubits
    inc_matrix = np.zeros((size, size))
    for i in range(size):
        inc_matrix[i, (i - 1) % size] = 1
    return inc_matrix

# Create a custom register operation for the incrementer
cudaq.register_operation("INC", incrementer(num_qubits))

@cudaq.kernel
def check_incrementer_kernel():
    qubits = cudaq.qvector(4)
    INC(qubits[0], qubits[1], qubits[2], qubits[3])

# Print the INC circuit
print(cudaq.draw(check_incrementer_kernel))

result = cudaq.sample(check_incrementer_kernel, shots_count=1000).most_probable()
print('Incrementer kernel |0000> -> |{}>'.format(result))

### Exercise 2:
Using the fact that DEC` is the inverse of $\texttt{INC}$, create a kernel for a custom $\texttt{DEC}$ operation. Define a kernel on 4 qubits for the DEC operation that maps $\ket{x}$ to $\ket{x-1}$ mod 16 and verify that it works as expected for $\ket{0001}$.



In [ ]:
# Define the decrementer matrix

def decrementer(num_qubits):
    size = 2**num_qubits
    dec_matrix = np.zeros((size, size))
    for i in range(size):
        dec_matrix[i, (i + 1) % size] = 1
    return dec_matrix


# EDIT THE CODE BELOW THIS LINE

# Create a custom register operation for the decrementer called DEC


# EDIT THE CODE ABOVE THIS LINE

# Create a kernel that applies the DEC to the 4-qubit state |0001>
@cudaq.kernel
def check_decrementer_kernel():
    qubits = cudaq.qvector(4)
    # Initialize the qubits to |0001>
    x(qubits[3])
    # Apply the decrementer operation
    DEC(qubits[0], qubits[1], qubits[2], qubits[3])

result = cudaq.sample(check_decrementer_kernel, shots_count=1000).most_probable()
print('Decrementer kernel |0001> -> |{}>'.format(result))

### Avoiding Walking in Circles

It is important to note that, as currently outlined, we are on track to define a quantum walk that will allow a walker to move between positions $\ket{0}=\ket{0000}$ and $\ket{15}=\ket{1111}$ in one step.  This behavior might not be desired, for instance, in a financial application where the positions of the walker represent stock prices and movement is related to positive or negative investor sentiment.

To prevent one-step transitions between $\ket{0}$ and $\ket{1}$ , we use an auxiliary qubit (`endpoint_qubit`), Toffoli gates, CNOT gates, and a mid-circuit `reset` to ensure a coin in the state $\ket{1}$ does not trigger a shift right (`INC`) when the walker is at $\ket{1111}$, and a coin in the state $\ket{0}$ does not trigger a shift left (`DEC`) when the walker is in the state $\ket{0000}$. This process is depicted below for one step of the DTQW:

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/DTQW_endpoint.png?raw=1" width="600">

The corresponding functions to prevent one-step transitions between $\ket{0000}$ and $\ket{1111}$ are defined below.

In [ ]:
# Define kernels to prevent transitions between |0000> and |1111>

# Kernel to change a coin from 1 to 0 if the walker is in the state |1111>
@cudaq.kernel
def no_INC_at_right_endpoint(walker_qubits : cudaq.qvector, coin_qubit : cudaq.qubit, right_endpoint: cudaq.qubit):

    # Test if the coin is in |1> and the walker state is |1111>,  if so, change the end_point qubit to 1
    x.ctrl([coin_qubit, walker_qubits[0], walker_qubits[1], walker_qubits[2], walker_qubits[3]], right_endpoint)

    # Flip the state of the coin if the endpoint is triggered
    x.ctrl(right_endpoint, coin_qubit)


# Kernel to change a coin from 0 to 1 if the walker is in the state |0000>
@cudaq.kernel
def no_DEC_at_left_endpoint(walker_qubits : cudaq.qvector, coin_qubit : cudaq.qubit, left_endpoint: cudaq.qubit):
    # Bit Flip the walker and coin qubits
    x(coin_qubit)
    x(walker_qubits)

    # Trigger the left_endpoint qubit if the walker and the coin are now in the states |1> and |1111> respectively
    x.ctrl([coin_qubit, walker_qubits[0], walker_qubits[1], walker_qubits[2], walker_qubits[3]], left_endpoint)

    # Undo the bit flip on the walker and coin qubits
    x(walker_qubits)
    x(coin_qubit)

    # Flip the coin fron |0> to |1> if the endpoint qubit is triggered
    x.ctrl(left_endpoint, coin_qubit)

# Kernel to reset the coin and endpoint qubit
@cudaq.kernel()
def reset_coin_and_endpoint(coin_qubit : cudaq.qubit, endpoint: cudaq.qubit):
    # change the coin qubit back if it was targeted by the endpoint qubit
    x.ctrl(endpoint, coin_qubit)

    # reset the endpoint qubit to |0>
    reset(endpoint)

#### The State of the Coin

Not only is the position of the walker at any given time step a linear combination of the computational basis states, but also the coin is a linear combination of heads:

$$ \ket{0} = \begin{pmatrix}1 \\ 0 \end{pmatrix}$$

and tails:

$$ \ket{1} = \begin{pmatrix}0 \\ 1 \end{pmatrix}$$

and depends on $t$: $$\ket{\psi_C(t)} = \beta_0(t)\ket{0}+\beta_1(t)\ket{1},$$ for $\beta_0,\beta_1\in\mathbb{C}$ with $|\beta_0(t)|^2 + |\beta_1(t)|^2 = 1.$  For example, we might begin the experiment with a coin in the zero (heads) state: $\ket{\psi_C(0)}=\ket{0}$.  Another option would be to start the experiment with the coin in a state of superposition of both heads and tails: $\ket{\psi_W(0)}=\frac{1}{\sqrt{2}}\ket{0}+\frac{1}{\sqrt{2}}\ket{1}\equiv\ket{+}$.

#### Changing the State of the Coin

The flipping of the coin is carried out by a quantum operation, $F$.  If the coinflip operation is the bitflip operation, $X$,
and the initial state of the coin is $\ket{\psi_C(0)} = \ket{0}$, then, flipping the coin would just change the state from $\ket{0}$ to $\ket{1}$ &mdash; much like the classical random walk.   

A common and more interesting quantum coinflip operation is the Hadamard operator, $F=H$, as illustrated in the diagrams of the DTQW above.
Applying this operator iteratively to the coin in the initial state $\ket{\psi_C(0)} = \ket{0}$, will result in alternating coin states $\ket{+}$ and $\ket{0}$.

We're not limited to constant coin operators, and could consider a coin operator that depends on parameters such as the Grover operator used for searching a graph for marked vertices ([Li and Sun](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.6.033042), or for an exposition [Wong](https://arxiv.org/pdf/2011.14533)). Moreover, the coin operator might depend on parameters that change from one step to the next and can be learnable, as we'll see in the next tutorial.

#### Changing the State of the Walker Based on Coin Flips:

Now, we're ready to put all of these kernels together to program one step of the DTQW.  Let's set the initial state of the walker to be $\ket{\psi_W(0)} = \frac{1}{\sqrt{2}}(\ket{0010}+\ket{0011})$. The steps $S_+$ and $S_-$, which depend on the state of the coin, can be modeled with controlled-gates and the `INC` and `DEC` operations, respectively.  We've defined the $S_+$ step below.  How would you edit the code to call up the `DEC` operator when the coin qubit is in the $\ket{0}$ state?

### Exercise 3:
Complete the code below to define $\texttt{S-}$. $\texttt{S-}$ will apply the DEC operation to the walker qubits when the coin qubit is in the $\ket{0}$ state.



In [ ]:
# Set the number of qubits
num_qubits = 4
@cudaq.kernel
def DTQW_one_step(num_qubits: int):
    walker_qubits = cudaq.qvector(num_qubits)
    coin_qubit = cudaq.qubit()
    endpoint_qubit = cudaq.qubit()

    # Initial walker state 1/(sqrt{2}) ( |2>+|3>)
    initial_position(walker_qubits)

    # Initial coin state
    h(coin_qubit) #Comment to set the initial coin state to be |0>

    # One quantum walk step
    # Coin operation F=H
    h(coin_qubit)

    # Walker's position change

    ## Shifting right

    # Avoid shifting from |1111> to |0000> in case the coin is |1> by flipping the coin to |0>
    no_INC_at_right_endpoint(walker_qubits, coin_qubit, endpoint_qubit)

    # Shift right (S+) when the coin is |1> and the walker is not in the |1111> state
    INC.ctrl(coin_qubit, walker_qubits[0], walker_qubits[1], walker_qubits[2], walker_qubits[3])

    # Reset the coin and endpoints in case they were changed to avoid moving from |1111> to |0000>
    reset_coin_and_endpoint(coin_qubit, endpoint_qubit)

    ## Shifting left

    # Avoid shifting from |0000> to |1111> in case the coin is |0> by flipping the coin to |1>
    no_DEC_at_left_endpoint(walker_qubits, coin_qubit, endpoint_qubit)

    # Shift left (S-) when the coin is |0>
    # EDIT CODE BELOW THIS LINE


    # EDIT CODE ABOVE THIS LINE

    # Reset the coin and endpoints in case they were changed to avoid moving from |0000> to |1111>
    reset_coin_and_endpoint(coin_qubit, endpoint_qubit)


# Visualize the kernel for the quantum walk
#print(cudaq.draw(DTQW_one_step, num_qubits))

result = cudaq.sample(DTQW_one_step, num_qubits, shots_count=1000)

# defines indices for walker_qubits_register
walker_register = list(range(0, num_qubits))

walker_register_marginal_counts = result.get_marginal_counts(walker_register)

print(walker_register_marginal_counts)

plot_results(walker_register_marginal_counts, num_qubits)

Does the output match what you expect?  Experiment with different initial states of the coin and walker qubits and different coin operators.

# Learnable Quantum Walk in Financial Simulation


Efficiently loading classical data into quantum states is a fundamental challenge in quantum computing, underpinning a wide range of applications from quantum machine learning to financial modeling. For instance, encoding a probability distribution such as the historical closing prices of a stock over a fixed time period can be particularly demanding. In this notebook, you will adapt the Discrete Time Quantum Walk into a variational multi-Split Step Quantum Walk, capable of modeling the trading patterns of multiple investors for a specific stock. You will then optimize the walk’s parameters to effectively load the given probability distribution into a quantum state.


We'll reuse some code from previous section which we've copied in the code block below. This block contains a helper function for graphing the results of the SSQW and the functions to prevent one-step transitions between the states $\ket{0000}$ and $\ket{1111}$.

Make sure to execute the cell below.

In [ ]:
# Define kernels to prevent transitions between |0000> and |1111>

# Kernel to change a coin from 1 to 0 if the walker is the state |1111>
@cudaq.kernel
def no_INC_at_right_endpoint(walker_qubits : cudaq.qvector, coin_qubit : cudaq.qubit, right_endpoint : cudaq.qubit):

    # Test if the coin is in |1> and the walker state is |1111>,  if so, change the end_point qubit to 1
    x.ctrl([coin_qubit, walker_qubits[0], walker_qubits[1], walker_qubits[2], walker_qubits[3]], right_endpoint)

    # Flip the state of the coin if the endpoint is triggered
    x.ctrl(right_endpoint, coin_qubit)


# Kernel to change a coin from 0 to 1 if the walker is in the state |0000>
@cudaq.kernel
def no_DEC_at_left_endpoint(walker_qubits : cudaq.qvector, coin_qubit : cudaq.qubit, left_endpoint : cudaq.qubit):
    # Bit Flip the walker and coin qubits
    x(coin_qubit)
    x(walker_qubits)

    # Trigger the left_endpoint qubit if the walker and the coin are now in the states |1> and |1111> respectively
    x.ctrl([coin_qubit, walker_qubits[0], walker_qubits[1], walker_qubits[2], walker_qubits[3]], left_endpoint)

    # Undo the bit flip on the walker and coin qubits
    x(walker_qubits)
    x(coin_qubit)

    # Flip the coin fron |0> to |1> if the endpoint qubit is triggered
    x.ctrl(left_endpoint,coin_qubit)

# Kernel to reset the coin and endpoint qubit
@cudaq.kernel()
def reset_coin_and_endpoint(coin_qubit : cudaq.qubit, endpoint: cudaq.qubit):
    # change the coin qubit back if it was flipped to prevent transitions
    # between |0000> and |1111>
    x.ctrl(endpoint, coin_qubit)

    # reset the endpoint qubit to |0>
    reset(endpoint)

Below we've modified some of the code from previous section  that we will repurpose here.   In particular, we'll replace the `INC` and `DEC` operations for shifting the walker's position with the kernels `inc` and `dec` that are defined with gate operators as opposed to unitary operators.  The advantage of this approach is that we'll be able to create nested kernels that will simplify the exposition.  The code below works for a 4-qubit walk.  At the end of this lab you'll be tasked with adapting all the code to work for any number of qubits.

In [ ]:
# Define a kernel on 4 qubits for the inc operation that
# maps |x> to |x+1> mod 16
num_qubits = 4

@cudaq.kernel
def inc(qubits : cudaq.qview):
    x.ctrl([qubits[3], qubits[2], qubits[1]], qubits[0])
    x.ctrl([qubits[3], qubits[2]], qubits[1])
    x.ctrl(qubits[3], qubits[2])
    x(qubits[3])

# Define a kernel on 4 qubits for the dec operation that
# maps |x> to |x-1> mod 16
@cudaq.kernel
def dec(qubits : cudaq.qview):
    cudaq.adjoint(inc, qubits)



We also use the helper function  below to plot histograms of the results of the quantum walks.

In [ ]:
def plot_walk_results(result, num_qubits, title):
    """Function that plots a historgram of the results of a quantum walk

    Parameters
    ----------
    results: cudaq.SampleResult
        results dictionary of sampling a quantum walk
    num_qubits: int
        number of qubits in the quantum walk
    title: str
        title for the histogram
    """
    # Define a dictionary of results from the sampling
    # Initialize the dictionary with all possible bit strings of length 4 for the x axis
    result_dictionary = {}

    # Generate all possible bit strings of length 4
    for i in range(2**num_qubits):
        bitstr = bin(i)[2:].zfill(num_qubits)
        result_dictionary[bitstr] = 0

    # Update the results dictionary of results from the circuit sampling
    for k,v in result.items():
        result_dictionary[k] = v

    # Convert the dictionary to lists for x and y values
    x = list(result_dictionary.keys())
    y = list(result_dictionary.values())

    # Create the histogram
    plt.bar(x, y, color='#76B900')

    # Add title and labels
    plt.title(title)
    plt.xlabel("Positions")
    plt.ylabel("Frequency")

    # Rotate x-axis labels for readability
    plt.xticks(rotation=45)

    # Show the plot
    plt.tight_layout()
    plt.show()

---
## Introduction to the Context and Problem

### The Context

A significant challenge in quantum computing is the efficient encoding of classical data into quantum states that can be processed by quantum hardware or simulated on classical computers. This is particularly crucial for many financial applications, where the first step of a quantum algorithm often involves efficiently loading a probability distribution. For instance, to enable lenders to price loans more accurately based on each borrower's unique risk profile, it is essential to estimate individual loan risk distributions (such as default and prepayment) while accounting for uncertainties in modeling and macroeconomic factors. Efficiently implementing this process on a quantum computer requires the ability to load a log-normal or other complex distribution into a quantum state [(Breeden and Leonova)](https://www.tandfonline.com/doi/full/10.1080/01605682.2022.2115415).

Financial markets exhibit complex probability distributions and multi-agent interactions that classical models struggle to capture efficiently. Quantum walks can be used to generate probability distributions of market data. The quantum walk approach offers:
* Flexible modeling of price movements
* Better representation of extreme events compared to classical methods
* Ability to capture asymmetric return distributions [(Backer et al)](https://arxiv.org/pdf/2403.19502).


### The Problem

This tutorial explores the multi-split-step quantum walks (mSSQW) technique, as detailed by [Chang et al.](https://arxiv.org/pdf/2302.12500), to load a specific probability distribution into a quantum state.  Although this technique is applicable to various distributions, our focus here is on the log-normal distribution, which can model the spot price of a financial asset at maturity. In brief, the logarithm of the asset price follows:

$$
\log S_T \sim \mathcal{N} \left( \left( r - \frac{1}{2} \sigma^2 \right) T + \log S_0, \sigma^2 T \right)
$$

Where:
- $S_T$ → Asset price at time T.  
- $S_0$ → Initial asset price.  
- $T$ → Time to maturity (future time period).  
- $r$ → Risk-free interest rate.  
- $\sigma$ → Volatility (measure of price fluctuation).

Below, you will find a graph illustrating the sampling of a continuous log-normal distribution targeted in this tutorial.

In [ ]:
# Prepare data

S = 4.0  # initial spot price
vol = 0.3  # volatility (40%)
r = 0.06  # annual interest rate (4%)
T = 30 / 365  # 90 days to maturity


def prepare_data(S, vol, r, T, N=2**10, num_qubits=4):
    mu = (r - 0.5 * vol**2) * T + np.log(S)
    sigma = vol * np.sqrt(T)
    mean = np.exp(mu + sigma**2 / 2)
    std_dev = np.sqrt((np.exp(sigma**2) - 1) * np.exp(2 * mu + sigma**2))

    # lowest and highest value considered for the spot price
    low, high = np.maximum(0, mean - 3 * std_dev), mean + 3 * std_dev

    # Sample from the log-normal distribution
    np.random.seed(42)
    log_normal_samples = np.clip(np.round(np.random.lognormal(mean=mu, sigma=sigma, size=N) * 2) / 2, 0, 2**num_qubits)

    # Calculate and return the target distribution from log-normal distribution
    bins = np.arange(2**num_qubits + 1)
    target = np.bincount(log_normal_samples.astype(int), minlength=len(bins) - 1)
    return target / target.sum()


target = prepare_data(S, vol, r, T)

# Plot the target distribution
fig, ax1 = plt.subplots(figsize=(20, 5))  # Increased figure width to accommodate all ticks

# Quantum walk vs. target distribution comparison
x = np.arange(len(target))
ax1.bar(x, target, color='#00BFFF', alpha=0.8)
ax1.set(title='Target Distribution', xlabel='Spot Price at Maturity $S_T$', ylabel='Probability')
ax1.set_xticks(np.arange(len(target)))
ax1.tick_params(axis='x', which='major', labelbottom=True)
ax1.grid(alpha=0.3)

plt.tight_layout()  # Use this to avoid labels and titles being cut off
plt.show()

---
## Split Step Quantum Walk

The aim of this tutorial is to load the targeted distribution, graphed above, into a quantum state using quantum walks.  Before getting into the details of the mSSQW, let's first define the *[**split-step quantum walk (SSQW)**](https://arxiv.org/pdf/2302.12500)*.

SSQW is an extension of the DTQW introduced in previous section.  For the SSQW, each time step is split into two actions determined by the outcome of two coin flips, hence the name "split step." As was the case in the DTQW, instead of classical coins that can be in the discrete states heads or tails, we use single-qubit quantum state $\ket{\psi_{C}}$ to represent the state of the coin.  

In contrast to the DTQW, the SSQW involves flipping the coin twice for each time step. The first flip determines whether the state of the walker shifts right, and the second flip determines whether the state of the walker shifts left.  That is, we will apply a quantum coinflip operation $F_1$ on $\ket{\psi_{C}}.$ The outcome of this flip $F_1\ket{\psi_{C}}$ will determine whether the walker's state shifts to the right or remains unchanged. Next, during this time step, we apply another, possibly different, quantum operation $F_2$ on the same coin. This outcome $F_2(F_1\ket{\psi_{C}})$ determines whether the walker's state shifts to the left or remains unchanged.  The circuit diagram for one time step of the SSQW algorithm is given below:

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/SSQW.png?raw=1" width="400">

Additionally, since our goal is to model financial data, we must avoid scenarios where the walker with one coin flip moves directly between positions $\ket{0000}$ and $\ket{1111}$. Such transitions might suggest an investor with positive sentiment about the stock’s value encoded as $\ket{1111}$ would sell it at a very low price near the value encoded as $\ket{0000}$.  To prevent this, we use an auxiliary qubit `endpoint_qubit` as we did in Part 1. One step of the SSQW with an additional endpoint qubit is depicted as follows:

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/SSQW_endpoint.png?raw=1" width="800">

### Exercise  4:
Your next exercise is to code one step of the SSQW below using two different coin operations $F_1 = X$ and $F_2 = H$ and disallowing movement of the walker between position $\ket{0000}$ and $\ket{1111}$.


In [ ]:
# Set the number of qubits Do Not Change without changing inc and dec
num_qubits = 4

# Set the number of steps
num_time_steps = 1 #CHANGE_ME

# Kernel for one step of the SSQW without measurement
@cudaq.kernel
def SSQW_one_step(walker_qubits : cudaq.qview, coin_qubit : cudaq.qubit, endpoint_qubit : cudaq.qubit):

    # EDIT CODE BELOW THIS LINE

    # One quantum walk step

    # First flip of the coin with the first coin operation, F1

    # Second flip of the coin with the second coin operation, F2

    # Don't forget to reset the qubits that may have been used to prevent transitions between |0000> and |1111>


    # EDIT CODE ABOVE THIS LINE

@cudaq.kernel()
def SSQW_with_measurement(num_qubits : int, num_time_steps : int):
    walker_qubits = cudaq.qvector(num_qubits)
    coin_qubit = cudaq.qubit()
    endpoint_qubit = cudaq.qubit()

    # Initial walker state 1/sqrt(2)(|1000> + |0000>) #CHANGE_ME
    #h(walker_qubits[0])
    #x(walker_qubits)


    # Initial coin state    #CHANGE_ME
    x(coin_qubit)

    for _ in range(num_time_steps):
        SSQW_one_step(walker_qubits, coin_qubit, endpoint_qubit)




#cudaq.draw(SSQW_with_measurement, num_qubits, num_time_steps)

# Sample the kernel for the quantum walk
result = cudaq.sample(SSQW_with_measurement, num_qubits, num_time_steps, shots_count=5000)

# defines indices for walker_qubits_register
walker_register = list(range(0, num_qubits))

walker_register_marginal_counts = result.get_marginal_counts(walker_register)


title = 'Sampling SSQW with {} time steps'.format(num_time_steps)
plot_results(walker_register_marginal_counts, num_qubits, title)

Once you've completed the exercise above, feel free to experiment with different combinations of the following:

* initial states of the coin
* initial states of the walker
* coin operations (e.g., `h`, `t`, `z`, etc.)
* `num_time_steps`

For instance, you might select initial states and coin operators to verify that regardless of the coin, a walker beginning in state $\ket{0000}$ will not move to $\ket{1111}$ in one time step:
<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/SSQW_endpoint_verified.png?raw=1" width="800">

---
## Multi-Split Step Quantum Walk

As you might have noticed in the previous section, by varying the initial state of the walker and coin, and experimenting with different coin flip operations, we can generate many different distributions.  Another variation to consider is to change the coin operations, $F_1$ and $F_2$, throughout the walk.  For example, perhaps we start with two bit flip operations to change the state of the coin in time step one and then switch to two Hadamard operations on the coin in time step 2.  Go ahead and edit the code block above to observe the newly generated probability distribution.  

The challenge, and the focus of the remainder of this tutorial, is to identify initial states and coins that will generate a targeted probability distribution. To keep things simple, we'll fix the initial states of the coin and the walker. Using parameterized coin operations, we can employ a classical optimizer to iteratively evolve the system to generate the desired distribution.

What we've just described is the ***multi-Split-Step Quantum Walk (multi-SSQW)*** framework, which we have depicted below for a one time step. To adapt this for multiple time steps, the multi-SSQW quantum circuit is replaced with one made up of `num_time_steps` copies of the block sequence of the $k$ investor's split steps.   In a financial application, the different coin operations at each time step can represent distinct investors' decision-making processes reflecting positive or negative sentiments about the value of the stock. The investor's actions can affect a stock price that is being modeled by the probability distribution ([Chang et al.](https://arxiv.org/pdf/2302.12500)).

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/multi-SSQW_enpoint.png?raw=1" width="1200">

We've encoded the multi-SSQW circuit below using the [universal three-parameter gate](https://nvidia.github.io/cuda-quantum/latest/api/default_ops.html#u3), `u3`, for the coin flip operators.

In [ ]:
# Initialize the quantum walk kernel and position shift
# Fixed variables do not change
num_qubits = 4 # Number of qubits representing the state of the walker

# Variables that you can experiment with
num_investors = 2 # Number of investors
num_time_steps = 2 # Number of repetitions of the Investors' walk
shots_count = 10000

@cudaq.kernel
def multiSSQW_kernel(num_qubits: int, num_investors: int, num_time_steps : int, param : list[float]):
    """ kernel for the multi-SSQW circuit for num_investors and num_time_steps
    Parameters
        num_qubits : int
        Number of qubits for the walker's state

        num_investors : int
        Number of split-step pairs in one time step

        num_time_steps : int
        Number of iterations of the investor's split steps

        param : list[float]
        Parameters for the coin operators
    """
    walker_qubits = cudaq.qvector(num_qubits)
    coin_qubit = cudaq.qubit()
    endpoint_qubit = cudaq.qubit()
    # Initial walker state |0101>
    x(walker_qubits[1])
    x(walker_qubits[3])

    # Initial coin state (optional)
    u3(param[0], param[1], param[2], coin_qubit)

    for _ in range(num_time_steps):
        # Quantum walk split steps of all the investors
        for k in range(num_investors):
            # Split step of one investor


            # First coin operation
            u3(param[(k+1)*6-3], param[(k+1)*6-2], param[(k+1)*6-1], coin_qubit)

            # Walker's position change
            # Avoid shifting from |1111> to |0000> in case the coin is |1> by flipping the coin to |0>
            no_INC_at_right_endpoint(walker_qubits, coin_qubit, endpoint_qubit)
            # Shift to the right if the coin is |1> and the position is not |1111>
            cudaq.control(inc, coin_qubit, walker_qubits)

            # Reset the coin and endpoints in case they were changed to avoid moving from |1111> to |0000>
            reset_coin_and_endpoint(coin_qubit, endpoint_qubit)


            # Second coin operation
            u3(param[(k+1)*6], param[(k+1)*6+1], param[(k+1)*6+2], coin_qubit)

            # Controlled-decrementer
            # Avoid shifting from |0000> to |1111> in case the coin is |0> by flipping the coin to |1>
            no_DEC_at_left_endpoint(walker_qubits, coin_qubit, endpoint_qubit)

            # Apply the DEC if the the coin is in |0>
            x(coin_qubit)
            cudaq.control(dec, coin_qubit, walker_qubits)
            x(coin_qubit)

            # Reset the coin and endpoints in case they were changed to avoid moving from |0000> to |1111>
            reset_coin_and_endpoint(coin_qubit, endpoint_qubit)


The code block below defines functions for the classical optimization in the multi-SSQW hybrid workflow. We've selected the COYBLA classical optimizer and the mean square error (MSE) cost function for this example, but other optimizers and cost functions could easily be substituted.

In [ ]:
def quantum_walk_simulation(kernel, num_qubits, num_investors, num_time_steps, param, shots_count=shots_count):
    """
    Execute the multi-SSQW kernel with the given parameters and return probabilities.

    Parameters
        kernel : cudaq.kernel parameterized kernel to be sampled
        num_qubits, num_investors, num_time_steps: parameters defining the kernel
        param : float values for the gate parameters in the kernel
        shots_count : int, number of shots for sampling

    Returns
        probs : list
        A list of float probabilities sorted by bitstring (e.g., [P(00..0), ..., P(11..1)])
    """
    #    Sample the kernel (Measures all qubits by default)
    #    Returns a SampleResult object (dictionary of counts)
    full_result = cudaq.sample(kernel, num_qubits, num_investors, num_time_steps, param, shots_count=shots_count)

    #   Extract marginal counts for the first `num_qubits`
    #    This isolates the position register (indices 0 to num_qubits-1)
    #    and discards the states of coin or endpoint qubits.
    target_indices = list(range(num_qubits))
    marginal_counts = full_result.get_marginal_counts(target_indices)

    #    Initialize dictionary with ALL possible bitstrings
    #    This ensures the output list is fully populated (including 0s) and sorted.
    result_dictionary = {}
    for i in range(2**num_qubits):
        bitstr = bin(i)[2:].zfill(num_qubits)
        result_dictionary[bitstr] = 0

    #  Update with actual data from marginal counts
    for bitstr, count in marginal_counts.items():
        # Robust cleanup: remove any potential invisible characters/spaces
        clean_key = "".join(c for c in str(bitstr) if c in "01")

        if clean_key in result_dictionary:
            result_dictionary[clean_key] = count

    #   Convert counts to a list of probabilities
    #    Sort by keys to ensure the list order matches the histogram x-axis (000, 001, 010...)
    sorted_keys = sorted(result_dictionary.keys())
    probs = [result_dictionary[key] / shots_count for key in sorted_keys]

    return probs


def optimize_distribution(kernel, num_qubits, num_investors, num_time_steps, target, max_iter=100):
    """
    Optimize the quantum walk parameters to match the target distribution
    """
    errors = []
    best_error, best_params = float('inf'), None

    # Define the cost function as a mean square error between the results of the sampling of the kernel
    # with given parameters and the targeted distribution
    def cost_function(params):
        trained = quantum_walk_simulation(kernel, num_qubits, num_investors, num_time_steps, params)
        return np.mean((trained - target)**2)

    def callback(xk):
        errors.append(cost_function(xk))

    # Randomly initialize parameters
    random.seed(42)
    x0 = [random.uniform(0, np.pi) for _ in range(6*num_investors + 3)]


    # Optimize
    result = minimize(cost_function, x0=x0, method="COBYLA", options={'maxiter': max_iter}, callback=callback )
    if result.fun < best_error:
        best_error = result.fun
        best_params = result.x

    print(f"Current best error: {best_error}")

    return best_params, errors, best_error

Now, it's time to execute the multi-SSQW and plot the results!

In [ ]:
# Define the simulation target
cudaq.set_target("qpp-cpu") #for simulation on CPU
#cudaq.set_target("nvidia") #for simulation on GPU
# If you run on a GPU, up the shots_count=10000 for a more accurate simulation
# you can also increase the number of iterations to 200

# Optimize to get the final distribution
best_params, errors, final_error = optimize_distribution(multiSSQW_kernel, num_qubits, num_investors, num_time_steps, target)

# Simulate the quantum walk with the best parameters
qw = quantum_walk_simulation(multiSSQW_kernel, num_qubits, num_investors, num_time_steps, best_params)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Quantum walk vs. target distribution comparison
x = np.arange(len(target))
ax1.bar(x, qw, width=0.8, color='#76B900', alpha=0.7, label='Quantum Walk')
ax1.plot(x, target, '-o', color='#00BFFF', linewidth=2, markersize=8, label='Log-normal Distribution')
ax1.set(title='Distribution Comparison', xlabel='Spot Price at Maturity $S_T$', ylabel='Probability')
ax1.grid(alpha=0.3)
ax1.legend()

# Optimization errors plot
ax2.plot(errors, label='Error', color = '#00BFFF')
ax2.set(title=f'Optimization Error (Final: {final_error:.6f})', xlabel='Iteration', ylabel='Error')
ax2.grid(True)
ax2.annotate(f'Best Error = {final_error:.6f}', xy=(len(errors)-1, final_error), xytext=(-50, 20),
             textcoords='offset points', ha='center', fontsize=12, color='#76B900')

plt.tight_layout()
plt.show()

### Challange
Use the multi-split step quantum walk to model other distributions such as stock values over a period of time (you can access historical stock market data using packages such as <a href="https://pypi.org/project/yfinance/" style="font-weight: bold; text-decoration: underline;">yfinance</a>).You might notice that the success of this approach is very sensitive to the number of steps and the number of walkers. How might you edit the code above to treat these constants as additional parameters that can be optimized? For an even bigger challenge, rewrite the code above to work for an arbitrary number of qubits using the variable  `num_qubits`

**Future Directions:** Apply what you have learned here to adapt the code to create a [Quantum Walk-Based Adaptive Distribution Generator](https://arxiv.org/pdf/2504.13532) that will allow you to not only model financial data, but also model 2D data such as pixelized images of handwritten digits.

# Quantum Portfolio Optimization

In this part you will perform a fundamental financial task - portfolio optimization.  You will learn how to setup the optimization problem, solve it using two different quantum methods, add constraints, and explore methods for scaling up such approaches. This notebook will also explore the state-of-the-art Q-CHOP algorithm developed by Infleqtion (also contributors to this notebook) and JPMorgan Chase demonstrating how the fundamental examples you code up with CUDA-Q can be extended towards useful real-world applications.


------

## Introduction

Portfolio optimization is foundational for modern finance. The aim of portfolio optimization is to select the best combination of assets to maximize returns while minimizing risk. Qualitatively, this looks like the portfolio in the leftmost point of the "universe" of potential portfolios pictured below.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/portfolioopt.png?raw=1" alt="Graph of Portfolio Optimization" style="width: 900px;"/>


Seminal to this field is Harry Markowitz's Modern Portfolio Theory (MPT), introduced in the 1950s. Markowitz's groundbreaking work established the framework for diversification, demonstrating that the risk of a portfolio is not merely the sum of the risks of its individual assets. Instead, by carefully selecting a mix of assets with varying correlations, investors can reduce the overall portfolio risk.


Even with such insights, identifying good portfolios is extremely challenging when there are seemingly endless choices for investment selections.  Moreover, fund managers can add additional complexity by changing the weights of each portfolio or considering schemes where stock inclusion is not binary, but can include a number of different strategies.

Regardless of the setup, the biggest challenge for portfolio optimization is the combinatorial scaling of the universe of potential portfolios. It is well within the bounds of practicality to consider, for example, a portfolio constructed from 20 Fortune 100 stocks which is selected from 100 choose 20 or quintillions ($10^{20}$) of potential portfolios. This is where quantum computing might be able to help.

The idea is to encode the problem into a quantum state consisting of $n=100$ qubits. A clever algorithm could leverage the properties of superposition, entanglement, and interference to evolve this state towards a final state from which quality portfolios can be sampled with a high probability.  This avoids the need for brute force search of all $10^{20}$ portfolios and potentially provides a much more efficient means to sample optimal portfolios.

The simplest formulation of a portfolio optimization is an equal weighted quadratic unconstrained binary optimization (QUBO). The QUBO formulation is presented in the following equation.

$$\min_{x} (-\alpha x^T\mu + \beta x^T\sigma x) = \min_{x} (x^TQx) = \min_{x} (\sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i  x_j )$$

The $x$ vector is binary (1 or 0 entries) and is of length $n$ (total possible stocks) where the portfolio is indicated by the subset of 1 entries. The $\mu$ vector and $\sigma$ matrix contain information about the return of each stock and the covariance between pairs of stocks. This information is generally obtained from historical data which is an helpful but imperfect indicator of future behavior. Finally, the $\alpha$ and $\beta$ terms are  parameters that weigh the importance of risk and reward.

### Exercise  5:
Given the $\alpha$, $\beta$, $\mu$ and $\sigma$ values below, write code in the cell below to produce the QUBO matrix $Q$.  Note, in practice, the matrix is usually made upper diagonal by doubling the upper diagonal entries and zeroing out the lower diagonal entries.  This saves space rather than having two terms correspond to the same correlation.  Then, in following cell, write code to generate all possible portfolios and via brute force, determine the optimal portfolio.


In [ ]:
mu = np.array([0.80, 0.70, 0.25, 0.20])          # expected returns
Sigma = np.array([[0.90, 0.45, 0.55, 0.10],      # variances / covariances
                  [0.45, 0.40, 0.30, 0.08],
                  [0.55, 0.30, 0.25, 0.05],
                  [0.10, 0.08, 0.05, 0.05]])
alpha, beta = 1.0, 1.0
n = len(mu)

def portfolio_to_qubo(mu, Sigma, alpha=1., beta=1.):
    """
    Convert financial data into a QUBO Matrix

    Args:
        mu (np.array): length n vector of portfolio returns
        Sigma (np.array): nxn matrix of portfolio pair covariances
        alpha (float): tunable parameter for valuing return
        beta (float): tunable parameter for valuing risk


    Returns:
        Q (np.array): upper triangular QUBO Matrix
    """

    #TODO Start
    n = len(mu)
    Q = np.zeros((n, n))
    for i in range(n):
        Q[i, i] = #FIX_ME#
    for i, j in combinations(range(n), 2):
        Q[i, j] = #FIX_ME#
    return Q

Q = portfolio_to_qubo(mu, Sigma, alpha, beta)
print(Q)
#TODO End

Now, write a code that generates all of the possible portfolios, evaluates the portfolio quality by manually computing $x^TQx$, and returns the best portfolio.

In [ ]:
def evaluate_all_bitstrings(Q):
    """
    Evaluate QUBO objective x^T Q x for all possible bitstrings.

    Args:
        Q (np.ndarray): QUBO matrix (nxn)
        verbose (bool): Print all solutions if True

    Returns:
        best_x (np.array): Optimal bitstring
        best_val (float): Optimal objective value
        all_solutions (list): List of (bitstring, value) tuples
    """

    #TODO Start
    n = Q.shape[0]

    # Generate all possible bitstrings
    bitstrings = list(itertools.product([0, 1], repeat=n))

    # Evaluate each bitstring
    # Keep a running list of solutions and best and worst values
    solutions = []
    best_val = float('inf')
    best_x = None
    worst_val = float('-inf')
    worst_x = None

    # EDIT CODE BELOW THIS LINE

    # EDIT CODE ABOVE THIS LINE

    # Sort solutions by objective value
    solutions.sort(key=lambda x: x[1])

    print("\nBest solution:")
    print(f"x* = {best_x}")
    print(f"Objective value = {best_val:.6f}")

    print("\nWorst solution:")
    print(f"x* = {worst_x}")
    print(f"Objective value = {worst_val:.6f}")

    #print("\nAll solutions:")
    #print(solutions)

    return best_x, best_val, worst_x, worst_val, solutions

best_x, best_val, worst_x, worst_val, all_solutions = evaluate_all_bitstrings(Q)
#TODO End

Now, explore the interactive widget in the code cell below. It allows you to change the portfolio optimization parameters $\alpha$ and $\beta$.

Try the following scenarios:
- Set $\alpha=3$ and $\beta=1$
- Set $\alpha=1$ and $\beta=2$

Can you rationalize the optimal portfolios produced by these settings? (Hint: study the resulting mean return, $\mu$, and volatility, $\sigma$).

> **Note:** If the widget does not appear below, you can access it directly using [this link](https://nvidia.github.io/cuda-q-academic/quantum-applications-to-finance/images/QUBO_widget.html).

<iframe src="https://nvidia.github.io/cuda-q-academic/quantum-applications-to-finance/images/QUBO_widget.html" width="800" height="600"></iframe>



## From QUBO to Spin Operator Hamiltonian

The QUBO is a useful format because it is well understood how to map a QUBO to a quantum Hamiltonian whose ground state corresponds to the QUBO solution.  In the following sections you will explore two ways to find the ground state, first using the quantum approximate optimization algorithm (QAOA), and the second using an adiabatic approach and CUDA-Q dyanmics.

The first step to running either of these methods is to derive the Ising Hamiltonian, $H_C$, which corresponds to the QUBO problem. Starting with the QUBO definition $\sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i  x_j$, substitute the binary variables with spin variables so $x_i = \frac{1-z_i}{2}$. This results in

$$ \frac{1}{2} \sum_i Q_{ii} (1-z_i) +  \frac{1}{4} \sum_{i<j} Q_{ij} (1 - z_i - z_j +z_iz_j) $$
$$ \frac{1}{2} \sum_i Q_{ii} -\frac{1}{2} \sum_i Q_{ii}z_i +  \frac{1}{4} \sum_{i<j} Q_{ij}  - \frac{1}{4} \sum_{i<j} Q_{ij} z_i - \frac{1}{4} \sum_{i<j} Q_{ij}  z_j + \frac{1}{4} \sum_{i<j} Q_{ij}z_iz_j$$


Grouping the terms results in a constant $C$ that can be dropped as it has no impact on the optimization.

$$C=  \frac{1}{2} \sum_i Q_{ii} + \frac{1}{4} \sum_{i<j} Q_{ij}   $$

We can further simplify things by grouping the single variable terms and the interacting terms:
$$H_C =\sum_{i,j} J_{ij}\, z_i z_j + \sum_i h_i\, z_i,$$
 where $h_i$ are the coefficients of the single variable terms and $J_{ij}$ are the coefficients of the interacting terms:

$$ h_i  = \frac{-1}{2}Q_{ii}  - \frac{1}{4}\sum_{i<j} Q_{ij} - \frac{1}{4}\sum_{k<i} Q_{ki} $$

$$ J_{ij}  =  \frac{1}{4}Q_{ij}. $$



### Exercise  6:
Using this derivation, write two functions to 1) build an Ising Hamiltonian and 2) produce a CUDA-Q spin operator Hamiltonian. The first function should take an upper triangular QUBO matrix as a numpy array and return a list of coefficients for the single variable terms, a flattened list of pairs of indices for the double variable terms, and a list of coefficients for the double variable terms.  Returning each as a separate list in this way makes them easy to input to CUDA-Q kernels later.

Write a second function that uses these inputs and constructs the Hamiltonian as a CUDA-Q spin operator object. Print the Hamiltonian to confirm it is correct.


In [ ]:
def qubo_to_ising(Q, tol=1e-12):
    """
    Convert a QUBO matrix to an Ising Hamiltonian

    Args:
        Q (np.ndarray): QUBO matrix (nxn)
        tol (float): Cutoff for near zero terms

    Returns:
        h_list (list): List of coefficients for single variable terms
        pair_indices (list): Flattened list corresponding to the pairs of indices for off diagonal QUBO terms
        J_list (list): List of two variable term coefficients.
    """
    #TODO Start
    n = Q.shape[0]

    # two-body
    J_pairs, J_coeffs = [], []
    for i, j in combinations(range(n), 2):
        coeff = 0.25 * Q[i, j]
        if abs(coeff) > tol:
            J_pairs.extend([i, j])
            J_coeffs.append(float(coeff))

    # one-body
    h = np.empty(n, dtype=float)

    # diagonal contributions
    for i in range(n):

        h[i] += #FIX_ME#  # diagonal

        for j in range(n):     # linear z_i terms (sum terms in same row)

            if j > i:

                h[i] += #FIX_ME#

        for k in range(i):    # linear z_i terms (sum terms in same column)


            if k < i:

                h[i] += #FIX_ME#

    return h.tolist(), J_pairs, J_coeffs

h_list, pair_inds, J_list =    qubo_to_ising(Q)

print(h_list)
print(pair_inds)
print(J_list)


def ising_to_spinop(h, pair_inds, J, n_qubits):
    """
    Creates a CUDA-Q SpinOperator Hamiltonian

    Args:
        h_list (list): List of coefficients for single variable terms
        pair_indices (list): Flattened list corresponding to the pairs of indices for off diagonal QUBO terms
        J_list (list): List of two variable term coefficients.

    Returns:
        H (cudaq.spinop): SpinOperator QUBO Hamiltonian
    """
    H = cudaq.SpinOperator()
    for i, coeff in enumerate(h):
        if coeff: H += coeff * spin.z(i)
    for k, coeff in enumerate(J):
        i, j = pair_inds[2*k : 2*k+2]
        H += coeff * spin.z(i) * spin.z(j)
    return H

H = ising_to_spinop(h_list, pair_inds, J_list, n)
print(H)

#TODO END

## Gate-Based Approach - QAOA

The Quantum Approximate Optimization Algorithm (QAOA) is a hybrid quantum-classical approach to solving combinatorial optimization problems. To the $|\!+\rangle^{\otimes n}$ state, it applies alternating layers of two types of unitary operations &mdash; one driven by a cost Hamiltonian (what your defined in the previous exercise) and one driven by a mixer Hamiltonian, each controlled by adjustable parameters. Your cost Hamiltonian is written as:

$$
H_C = \sum_{i,j} J_{ij}\, Z_i Z_j + \sum_i h_i\, Z_i,
$$

The goal is to produce a state, that when sampled, produces bitstrings at or close to the ground state of the Hamiltonian (i.e., optimal portfolios).
The QAOA state after $p$ iterations (layers) is:
$$
|\psi(\boldsymbol{\gamma}, \boldsymbol{\beta})\rangle
= U_M(\beta_p)\,U_C(\gamma_p)\,\cdots\,U_M(\beta_1)\,U_C(\gamma_1)\,|\!+\rangle^{\otimes n},
$$

where:
- $U_C(\gamma_k) = e^{-i \gamma_k H_C}$ (cost unitary),  
- $U_M(\beta_k) = e^{-i \beta_k H_M}$ (mixer unitary, often chosen as a transverse-field Hamiltonian),  
- $|\!+\rangle^{\otimes n}$ is the uniform superposition over all computational basis states,  
- and $\gamma_k, \beta_k$ are the variational parameters over the $p$ layers.

A classical optimizer iteratively updates these parameters to minimize the following expectation value: $$\langle \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) | H_C | \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) \rangle.$$

### Exercise  7:
Code a QAOA kernel in CUDA-Q. Note how inputs are provided as lists in accordance with your earlier functions. The figure below shows the gates applied to two qubits and one layer of the QAOA circuit.
Specifically, the elements of the cost function are applied as $R_Z$ gates parameterized with a $\gamma$ for each layer, times $h_i$ for the single variable terms. The two variable terms are applied as a CNOT gate between the two corresponding qubits (controlled by the first), an $R_Z$ gate parameterized with the same $\gamma$ for each layer, times $J_{ij}$, followed by a repeat of the previous CNOT gate. The mixer terms are simple $R_X$ gates applied to each qubit parameterized with a $2.0*\beta$ for each layer.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/qaoa-subcircuit.png?raw=1" alt="QAOA subcircuit Diagram" style="width: 900px;"/>

In [ ]:
cudaq.set_target('qpp-cpu')

@cudaq.kernel
def qaoa_kernel(theta : list[float],
                qubit_count : int,
                layers : int,
                h_coeffs : list[float],
                pair_inds : list[int],
                J_coeffs : list[float]):

    """
    Creates a CUDA-Q kernel corresponding to a QAOA circuit

    Args:
        theta (list): list of the variational parameters
        qubit_count (int): number of qubits
        layers (int): number of qaoa layers
        h_list (list): List of coefficients for single variable terms
        pair_indices (list): Flattened list corresponding to the pairs of indices for off diagonal QUBO terms
        J_list (list): List of two variable term coefficients.

    Returns:
        (cudaq.kernel): QAOA kernel
    """

    #TODO Start
    q = cudaq.qvector(qubit_count)

    # Hadamards
    for idx in range(qubit_count):
        h(q[idx])

    for layer in range(layers):
        gamma = theta[layer]
        beta  = theta[layer + layers]

        # cost e^{-i γ H_C}
        # single-Z
        for i in range(qubit_count):
            coeff = h_coeffs[i]
            if coeff != 0.0:
                #FIX_ME#

        # ZZ
        num_pairs = len(J_coeffs)
        for k in range(num_pairs):
            i = pair_inds[2 * k]
            j = pair_inds[2 * k + 1]
            coeff = J_coeffs[k]

            #FIX_ME#

        # mixer e^{-i β ∑ X}
        for i in range(qubit_count):
            #FIX_ME#

     #TODO END

Test your kernel with the code below which computes an expectation value from the state produced by the circuit and feeds the result into a classical optimizer which optimizes the $\gamma$'s and the $\beta$'s. Then, the `shots` variable saves the results from sampling the circuit. Was the optimal portfolio the most probable?  

In [ ]:
import matplotlib.pyplot as plt


layers = 3
init = np.random.uniform(0, 0, 2*layers)

initial= cudaq.sample(qaoa_kernel, init, n, layers,  # Sample results from initial state with all 0 parameters
                     h_list, pair_inds, J_list, shots_count=10000)

print(initial)


cost_history = []

def qaoa_cost(theta):
    res = cudaq.observe(qaoa_kernel, H, theta, n, layers,
                        h_list, pair_inds, J_list)
    cost = res.expectation()
    cost_history.append(cost)

    return cost

opt_result = minimize(qaoa_cost, init, method='COBYLA',
                      options={'maxiter': 200, 'disp': True})


print(opt_result)

shots = cudaq.sample(qaoa_kernel, opt_result.x, n, layers,
                     h_list, pair_inds, J_list, shots_count=10000)

print(shots)
print('Most-probable bit-string :', np.array(shots.most_probable()).astype(str))

plt.figure(figsize=(10, 6))
plt.plot(cost_history)
plt.xlabel('Iteration')
plt.ylabel("Expectation Value")
plt.title('QAOA Optimization Progress')
plt.grid(True)
plt.show()

The QAOA result may not produce the optimal portfolio most frequently due to the *approximate* nature of QAOA.  However, the resulting QAOA state should sample the optimal portfolio with much higher probability than randomly sampling all bitstrings. Run the script below on the sample results and confirm that the optimal portfolio is at least in the top 5 results sampled from the QAOA procedure.

In [ ]:
from helper import top_k_bitstrings

# Enter a CUDA-Q Sampleresult, Q, and the number of top bistrings to list
top5 = top_k_bitstrings(shots, Q, k=5)

This variability can make it challenging to assess a particular optimization method. For example, what if you run QAOA and produce a state that only samples the second best portfolio?   If there are quintillions of portfolios, that is not too bad!  Result quality for simulated quantum algorithms is often quantified with a so called approximation ratio.  This ratio can be defined a few ways, but the idea is to see how close the result is to the optimal portfolio.

A pragmatic way to do this is take the best result from the sample and compare it to the optimal value. For a small problem like you completed, that will probably be trivially 1 as you sample the optimal state many times. A better estimate to assess the algorithm performance is the average sample result divided by the optimal value.  This approach can also help prove that the QAOA procedure really did help produce a state that samples better portfolios and that you did not just get lucky sampling the ground state.

To better visualize the performance of your QAOA function, use the `plot_samples_histogram` function to plot a sample from the QAOA kernel with your initial parameters (a uniform superposition as all variational parameters are 0) and the final state you produced.  We've labeled "Good Portfolios" as those results that correspond to a negative $H_C$ value. Did the algorithm work?  

Note, `plot_samples_histogram` takes the initial and final CUDA-Q sample results as inputs, as well as the list of all solutions output by the `evaluate_all_bitstrings` function you wrote previously.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from helper import plot_samples_histogram

fig, ax = plot_samples_histogram(initial, shots, all_solutions,
                                "Portfolio Optimization Results")
plt.show()

Assume you are a portfolio manager and you need to be confident that you are suggesting a portfolio that has more reward than risk (a negative cost function value in this case).  From the histogram, estimate the probability of choosing a good portfolio from the initial solution and your QAOA solution?  

Though the example in this lab is small, it can be easily scaled up.  In fact, the same approach used in "[Divide and Conquer MaxCut QAOA](https://github.com/NVIDIA/cuda-q-academic/tree/main/qaoa-for-max-cut)" can potentially be used here to divide the problem into sub QAOA problems which can be solved in parallel and then knit together to obtain the final results. However, this depends on the sparsity of the QUBO and can only be used if certain stocks have negligible covariance.

Another possible aporoach is to use AI.  Rather than performing the QAOA optimization to produce the circuit corresponding to the ground state, techniques like [QAOA-GPT](https://arxiv.org/abs/2504.16350) can use a generative model to sample QAOA circuits.  Such an approach has promise, but is highly dependent on quality training data such that generated circuits are good approximations for the ground state.

## Adiabatic Approach

An alternative to gate-based quantum computation is annealing. Annealing can be used to solve optimization problems encoded into a QUBO Hamiltonian like you have already done. The difference is that annealing takes advantage of a special Hamiltonian of the form

$$ H = (1-t)H_{initial} + (t)H_{objective} $$

The adiabatic theorem states that such a Hamiltonian will remain in its ground state if the system evolves slowly enough. Consider why this is advantageous here.  At $t=0$, the system Hamiltonian is exactly $H_{initial}$. In practice, $H_{initial}$ is selected to have a very easy to prepare ground state.  Avoiding costly state preparation routines enables such an approach to more easily run on quantum hardware.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/adiabatic.png?raw=1" alt="Diagram of Adiabatic Approach" style="width: 900px;"/>



Though easier said than done, if an appropriate annealing schedule is chosen, $H$ will evolve from $H_{initial}$ to a mixture of $H_{initial}$ and $H_{objective}$ to a final state corresponding to the ground state of $H_{objective}$. This state is the exact same as what QAOA is trying to approximate.

CUDA-Q can simulate such an evolution using its dynamics solvers which solve the differential equations that govern such an evolution. The code below will use annealing to find an optimal portfolio. The $H_{initial}$ is selected to be Pauli-$X$ operations on all qubits which corresponds to the easily prepared $\ket{+}^{\otimes n}$ state. The problem Hamiltonian, $H_{objective}$, is just the cost Hamiltonian that you used for QAOA.

Using the `cudaq.evolve` API, the simulation is run and produces the state below.  This state can be input into a kernel and sampled similarly to the final state from the QAOA result. Note, the `dynamics` backend needs to be set in order to run `evolve`, but the backend must switch back to `nvidia` to build and sample the resulting state.


### Exercise  8:
    
Confirm the adiabatic approach works. Notice how the variable $T$ controls how fast the evolution occurs. Try rerunning with $T=10$.  What happens to the quality of the result?



💻 Just a heads-up: The remainder of this notebook is designed to be run on an environment with a GPU. If you don't have access to a GPU, feel free to read through the cells and explore the content without executing them. Enjoy learning! ⭐

In [ ]:
from cudaq import spin, boson, ScalarOperator, Schedule, ScipyZvodeIntegrator

cudaq.set_target('dynamics')

dimensions = {0: 2, 1:2, 2:2, 3:2}

Hinitial = cudaq.SpinOperator()
for i in range(n):
    Hinitial += spin.x(i)

Htotal = (1 - ScalarOperator(lambda t: t/T)) * Hinitial -  ScalarOperator(lambda t: t/T) * H

T=100

n = 4                             # example size
amp = (1/np.sqrt(2**n)) * np.ones(2**n, dtype=np.complex128)
psi0 = cudaq.State.from_data(amp)

steps = np.linspace(0, T, 100000)

schedule = Schedule(steps, ["t"])


evolution_result = cudaq.evolve(
    Htotal,
    dimensions,
    schedule,
    psi0,
    observables=[H],
    collapse_operators=[],
    integrator=ScipyZvodeIntegrator())


print(evolution_result.final_state())
state = evolution_result.final_state()
c = np.array(state).tolist()

cudaq.set_target('nvidia')

@cudaq.kernel
def final_state(amplitudes: list[complex]):
    q = cudaq.qvector(amplitudes)


dynamics_final = cudaq.sample(final_state, c, shots_count =10000)

fig, ax = plot_samples_histogram(initial, dynamics_final, all_solutions,
                                "Portfolio Optimization Results")

## Adding Constraints


In practice, it is often helpful to constrain a portfolio optimization problem. Maybe a portfolio manager wants to only select a portfolio consisting of $k$ stocks rather than all possible combinations considered thus far.  This means that the there is now some range of feasible solutions where exactly $k$ stocks are selected.


The constraint can be easily added to the QUBO by adding a quadratic constraint term of the form $ (\sum_i x_i - k)^2$ to produce

$$ x^TQx = \sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i  x_j  + ( \sum_i x_i - k )^2$$

Expanding the constraint term results in:

$$
\begin{aligned}
( \sum_i x_i - k)^2 &= ( \sum_i x_i )^2 -2k \sum_i x_i + k^2 \\
& = ( \sum_i x_i^2 + \sum_{i\ne j}x_i x_j) -2k \sum_i x_i + k^2
\end{aligned}
$$

Because $x_i$ is binary, we can replace $\sum_i x_i^2$ with $\sum_i x_i$. So our expressions becomes:

$$
\begin{aligned} (\sum_i x_i - k)^2 & = \\
& = (\sum_i x_i + \sum_{i \ne j} x_i x_j)  -2k\sum_i x_i + k^2\\
& = (1-2k)\sum_i x_i + \sum_{i \ne j} x_i x_j + k^2
\end{aligned}$$


Generally, the constraint has a factor  $\lambda$ which is a tuneable parameter for the strength of the constraint. After working out the example, there is a $\lambda(1-2k)$ factor added to the diagonal terms, a $2\lambda$ term added to each off diagonal (the 2 comes from the upper diagonal QUBO form we are using), and a constant term of $\lambda k^2$, which is often omitted since it has no impact on the result.

### Exercise  9:
Write a new $\texttt{portfolio\_to\_qubo}$ function below which now adds constraints to restrict the portfolio solutions  to $k$ stocks. Using all of the functions created earlier, rerun an adiabatic simulation of this constrained result.  Does the histogram produced below indicate that the constraint worked?


In [ ]:
def portfolio_to_qubo_constrained(mu, Sigma, alpha=1., beta=1., lam =10, k = 2):
    """
    Convert financial data into a QUBO Matrix

    Args:
        mu (np.array): length n vector of portfolio returns
        Sigma (np.array): nxn matrix of portfolio pair covariances
        alpha (float): tunable parameter for valuing return
        beta (float): tunable parameter for valuing risk
        lam (float): tuanble penalty for constraint violation
        k (int): constrain to favor portfolios of k stocks

    Returns:
        Q (np.array): upper triangular QUBO Matrix
    """

    n = len(mu)
    Q = np.zeros((n, n))
    # EDIT CODE BELOW THIS LINE
    # Populate matrix Q with values

    # EDIT CODE ABOVE THIS LINE

    return Q

Q_con = portfolio_to_qubo_constrained(mu, Sigma, alpha, beta, lam =10, k =2)
print(Q_con)

best_x, best_val, worst_x, worst_val, all_solutions = evaluate_all_bitstrings(Q_con)


h_list_con, pair_inds_con, J_list_con =    qubo_to_ising(Q_con)

H_con = ising_to_spinop(h_list_con, pair_inds_con, J_list_con, n)



from cudaq import spin, boson, ScalarOperator, Schedule, ScipyZvodeIntegrator

cudaq.set_target('dynamics')

dimensions = {0: 2, 1:2, 2:2, 3:2}

Hinitial = cudaq.SpinOperator()
for i in range(n):
    Hinitial += spin.x(i)

Htotal = (1 - ScalarOperator(lambda t: t/T)) * Hinitial -  ScalarOperator(lambda t: t/T) * H_con

T=100

n = 4                             # example size
amp = (1/np.sqrt(2**n)) * np.ones(2**n, dtype=np.complex128)
psi0 = cudaq.State.from_data(amp)

steps = np.linspace(0, T, 10000)
schedule = Schedule(steps, ["t"])


evolution_result = cudaq.evolve(
    Htotal,
    dimensions,
    schedule,
    psi0,
    observables=[H_con],
    collapse_operators=[],
    integrator=ScipyZvodeIntegrator())


print(evolution_result.final_state())
state = evolution_result.final_state()
c = np.array(state).tolist()

cudaq.set_target('nvidia')

@cudaq.kernel
def final_state(amplitudes: list[complex]):
    q = cudaq.qvector(amplitudes)


dynamics_final_con = cudaq.sample(final_state, c, shots_count =10000)

fig, ax = plot_samples_histogram(initial, dynamics_final_con, all_solutions,
                                "Portfolio Optimization Results")

## Improving Adiabatic Convergence with Q-CHOP

Adding a constraint means that you no longer care about the entire universe of $2^n$ portfolios, but only a subset of feasible solutions.  Though the standard adiabatic approach always ends with a feasible ground state solution of the total Hamiltonian, it is not guaranteed to remain within the feasible space of solutions during the evolution. If somehow, the evolution process could remain within the feasible solution space, convergence could be much faster and potentially reach higher quality solutions.

The [Q-CHOP Algorithm](https://arxiv.org/abs/2403.05653) developed by Infleqtion and JPMorgan Chase does just that and provides a clever way to modify the evolution process to remain within the feasible solution space. Q-CHOP takes advantage of the fact that it is sometimes easy to prepare the worst feasible solution.  The Q-CHOP evolution starts with the Hamiltonian whose ground state is equal to the worst feasible state and slowly rotates it (within the subspace of feasible solutions) to the problem Hamiltonian, resulting in a ground state of the problem Hamiltonian. The advantage of this is a procedure that can both converge faster and produce higher quality results by ensuring the evolution remains within the feasible space of solutions.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-3-images/qchop_diagram.png?raw=1" alt="Diagram of the QChop Algorithm" style="width: 900px;"/>

An astute reader might rightfully question the fact that it is easier to prepare the worst feasible state for portfolio optimization problems.  This observation is correct: finding the worth feasible state is just as hard as finding the best possible state! The Q-CHOP developers circumvent this with a clever two-stage approach.  First, they run Q-CHOP from an arbitrary feasible state. This is trivial to prepare as you can select any state that satisfies the constraint.

Then, they use a modified objective function of the form $(H_{objective} - E_{initial})^2$.  This will drive the evolution towards *either* the best or worst feasible state. If the result is the best state, the problem is solved.  If it is the worst, a second Q-CHOP run can be initialized from it. Remarkably, even if Q-CHOP requires both stages, it can still converge faster and to higher quality solutions than the standard adiabatic approach.

The Q-CHOP code is proprietary, so it cannot be interacted with directly. However, the interactive widget in the code cell [here](https://nvidia.github.io/cuda-q-academic/quantum-applications-to-finance/images/Q_CHOP_Animated.html) will allow you to compare a Q-CHOP run against a standard adiabatic approach.

The widget demonstrates two scenarios:
* **Left Panel:** A case where Q-CHOP requires only a single run, as it lands on the best feasible state.
* **Right Panel:** A two-stage situation where the worst feasible state is found in the initial run.

Infleqtion used CUDA-Q dynamics to perform portfolio optimization on real financial data which is detailed in [Spotlight: Infleqtion Optimizes Portfolios Using Q-CHOP and NVIDIA CUDA-Q Dynamics](https://developer.nvidia.com/blog/spotlight-infleqtion-optimizes-portfolios-using-q-chop-and-nvidia-cuda-q-dynamics/). For instances of selecting portfolios of size 7 or 8 from 15 possible stocks, Q-CHOP on average could produce a solution within 99.5% of the optimal solution with just 70 samples. This is three orders of magnitude fewer samples than is required to search the 12,870 potential portfolios by random sampling and provides great promise for future larger scale runs on physical QPUs.

## Scaling Up Sampling

It is worth a brief mention of the fact that the QAOA and adiabatic approaches both require sampling a final state after the algorithm completes. When sampling on a phsyical QPU, this means a circuit must prepare the target state and measure it for every single shot. For simulation this is easy, especially for small problem sizes, but this can be very time consuming and limit the sampling that can be done on a QPU.  For larger portfolio optimization problems, there is a tradeoff between the number of samples taken and the best solution sampled.



### Exercise  10:

Using the code below, generate random $\mu$ and $\sigma$ for $n=15$ stocks. Use your previous functions to run a QAOA. Run four different samples of the final state with 50, 100, 500, and 1000 shots and check how many times the optimal portfolio is sampled.  For this rather modest problem size, is there risk it is missed?


In [ ]:
def generate_portfolio_params(n, seed=42):
    """
    Generate a random covariance matrix and mean vector for portfolio optimization

    Args:
    n: size of the matrix/vector
    seed: random seed for reproducibility

    Returns:
    mu (np.array): vector of lenght n of stock returns
    Sigma (np.array): n x n matrix of stock covariances
    """
    np.random.seed(seed)

    # Generate random expected returns (mu) between 0.1 and 1.0
    mu = np.random.uniform(0.1, 1.0, n)

    # Generate a random positive semi-definite covariance matrix
    # Method: Create a random matrix A, then Sigma = A * A^T
    A = np.random.randn(n, n)
    Sigma = np.dot(A, A.T)

    # Scale the covariance matrix to reasonable values
    Sigma = Sigma / np.max(Sigma) * 0.9  # Scale max to 0.9

    # Ensure diagonal elements are positive (variances)
    np.fill_diagonal(Sigma, np.abs(np.diag(Sigma)) + 0.05)

    return mu, Sigma


big_mu, big_sigma = generate_portfolio_params(n)

Q_big = portfolio_to_qubo(big_mu, big_sigma)

best_x_big, best_val_big, worst_x_big, worst_val_big, all_solutions_big = evaluate_all_bitstrings(Q_big)

h_list_big, pair_inds_big, J_list_big =    qubo_to_ising(Q_big)

H_big = ising_to_spinop(h_list_big, pair_inds_big, J_list_big, n)

In [ ]:
layers = 3

init = np.random.uniform(0, 0, 2*layers)

initial_big = cudaq.sample(qaoa_kernel, init, n, layers,
                     h_list_big, pair_inds_big, J_list_big, shots_count=1000)

def qaoa_cost(theta):
    res = cudaq.observe(qaoa_kernel, H_big, theta, n, layers,
                        h_list_big, pair_inds_big, J_list_big)
    return res.expectation()

opt_result = minimize(qaoa_cost, init, method='COBYLA',
                      options={'maxiter': 500, 'disp': True})

print(opt_result)

shots_big = cudaq.sample(qaoa_kernel, opt_result.x, n, layers,
                     h_list_big, pair_inds_big, J_list_big, shots_count=1000)

In [ ]:
#TODO START
# Produce the appropriate samples sizes and count how many have the optimal portfolio

shots_big_50 = cudaq.sample(#FIX_ME#)

shots_big_100 = cudaq.sample(#FIX_ME#)

shots_big_500 = cudaq.sample(#FIX_ME#)

shots_big_1000 = cuda)

best = "".join(best_x_big.astype(str))

print("\n Number of times best portfolio is sampled in 50 shots:", shots_big_50.count(best))
print("\n Number of times best portfolio is sampled in 100 shots:" , shots_big_100.count(best))
print("\n Number of times best portfolio is sampled in 500 shots:", shots_big_500.count(best))
print("\n Number of times best portfolio is sampled in 1000 shots:", shots_big_1000.count(best))
# TODO END

Notice when a small number of samples is run, it is quite possible for the optimal solution to be missed. Our example of $n=15$ is rather modest, and this problem likey gets worse with larger problem sizes.  So, every shot matters to improve the quality of the portfolios generated from such an application.  One way to handle situations like this in the future will be to parallelize sampling across multiple QPUs.

CUDA-Q is designed with the capabilities for running jobs in such a multi-QPU data center.  Using the `mqpu` backend, you can sample the kernel you produced from the 15 qubit QAOA run above across multiple simulated QPUs.  If you have access to multiple GPUs, try running the code below.  

This feature can help CUDA-Q users run simulations faster (if the original task is onerous enough) and lays the groundwork for parallelizing tasks like sampling to improve algorithm results at scale.

In [ ]:
import time

cudaq.set_target('nvidia', option='mqpu')

s = time.time()
result = cudaq.sample(qaoa_kernel, opt_result.x, n, layers, h_list_big, pair_inds_big, J_list_big, shots_count=8000000)
e = time.time()
print("Time to run in serial:",e-s)

start_time = time.time()
result_1 = cudaq.sample_async(qaoa_kernel, opt_result.x, n, layers, h_list_big, pair_inds_big, J_list_big, shots_count=2000000, qpu_id=0)
result_2 = cudaq.sample_async(qaoa_kernel, opt_result.x, n, layers, h_list_big, pair_inds_big, J_list_big, shots_count=2000000, qpu_id=1)
result_3 = cudaq.sample_async(qaoa_kernel, opt_result.x, n, layers, h_list_big, pair_inds_big, J_list_big, shots_count=2000000, qpu_id=2)
result_4 = cudaq.sample_async(qaoa_kernel, opt_result.x, n, layers, h_list_big, pair_inds_big, J_list_big, shots_count=2000000, qpu_id=3)
result_1.get()
result_2.get()
result_3.get()
result_4.get()
end_time = time.time()
print("Time to run in parallel:", end_time - start_time)

## Summary

You have now sucessfully learned how to use CUDA-Q to solve portfolio optimization problems and more generally, any problem that can be formulated as a QUBO.  You have learned two different approaches to solving such problem using methods like gate-based QAOA or adiabatic approaches simulated with CUDA-Q `dynamics` and explored state-of-the-art approaches like Q-CHOP to more efficiently solve QUBO problems.  


### The Next Step: Quantum Stochastic Walks
Looking ahead, the field of quantum finance is continually evolving, with new algorithms emerging that offer different perspectives on optimization. One such exciting area is Quantum Stochastic Walks (QSW).

Unlike the methods you've learned, which translate the portfolio problem into a QUBO, QSW approaches portfolio optimization as a dynamic process on a financial network. This model can be understood through the following analogy:

* The Landscape: Each stock is a node on a graph, and your investment capital is personified as a “walker” exploring this landscape.

* The Journey: The walker's movement between nodes represents rebalancing the portfolio. This journey is guided by a hybrid engine: coherent quantum evolution uses superposition to find diversified pathways, while classical diffusion penalizes moves between highly correlated assets.

* The Destination: The final portfolio weights are determined by the walker's long-term behavior. The probability of finding the walker at any given node becomes that stock's final allocation (for instance, a 10% probability translates to a 10% portfolio weight).

This approach has been shown to produce robust, highly diversified portfolios with significantly lower turnover and transaction costs ([Chang et al](https://arxiv.org/pdf/2507.03963)). For those interested in the intersection of network science, quantum dynamics, and finance, QSW offers an area for further exploration. It represents a conceptual shift from static optimization to a dynamic, network-based simulation, highlighting the diverse ways quantum mechanics can be applied to financial challenges.